# Notebook 02 - Model Training

**Project:** Network Guardian IDS
**Phase:** 3 of 4
**Sprint Day:** 5 of 10

This notebook runs the three training pipelines. All logic lives in
`src/models.py`. This notebook only orchestrates and reports.

**Do not modify any cell in this notebook.** If a cell needs a change,
raise a decision file per `docs/decision/Decision Template.md`.

Cell ownership:

| Cell | Content | Owner |
|---|---|---|
| 2-4 | Shared setup | All |
| 5 | Random Forest training | Amber (Role 3) |
| 6 | Random Forest verification | Amber (Role 3) |
| 7 | XGBoost training | Kenneth (Role 4) |
| 8 | XGBoost verification | Kenneth (Role 4) |
| 9 | TabNet training | Dwayne (Role 5) |
| 10 | TabNet verification | Dwayne (Role 5) |
| 11 | Cross-model comparison | Kit (Role 1) |
| 12 | Summary | Kit (Role 1) |


In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd()).resolve()
_repo_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

print("Repo root on sys.path:", _repo_root)
print("src package present  :", (_repo_root / "src" / "__init__.py").is_file())


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.models import (
    load_processed_arrays,
    encode_labels,
    train_rf,
    train_xgb,
    train_tabnet,
)
from src.utils import PATHS, RANDOM_STATE, set_seed

set_seed(RANDOM_STATE)

print("Repository root :", PATHS.repo_root)
print("Processed dir   :", PATHS.processed)
print("Models dir      :", PATHS.models)
print("Outputs dir     :", PATHS.outputs)
print("Seed            :", RANDOM_STATE)

X_train, y_train_str, X_test, y_test_str, class_order = load_processed_arrays()
y_train = encode_labels(y_train_str, class_order)
y_test  = encode_labels(y_test_str,  class_order)

print()
print("X_train    :", X_train.shape, X_train.dtype)
print("X_test     :", X_test.shape, X_test.dtype)
print("y_train    :", y_train.shape, y_train.dtype)
print("y_test     :", y_test.shape, y_test.dtype)
print("class_order:", class_order)


In [ ]:
assert class_order == ["normal", "dos", "probe", "r2l", "u2r"], (
    f"Class order mismatch: {class_order}"
)
print("Class order verified. Ready for training.")


In [ ]:
# Random Forest training (Amber, Role 3)
# Contract: docs/contracts/role-3-rf-architecture-spec.md
# Estimated wall time: 1.5 to 3 hours

rf_result = train_rf(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
    n_trials=20,
    n_folds=5,
)
print("Random Forest CV Macro F1:", rf_result["cv_mean_macro_f1"])


In [ ]:
# Random Forest artifact verification (Amber, Role 3)

assert (PATHS.models / "rf_baseline.joblib").is_file(), "RF model missing"
assert (PATHS.models / "rf_baseline.meta.json").is_file(), "RF meta missing"
print("Random Forest artifacts verified.")


In [ ]:
# XGBoost training (Kenneth, Role 4)
# Contract: docs/contracts/role-4-training-execution-spec.md
# Estimated wall time: 4 to 6 hours

xgb_result = train_xgb(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
    n_trials=30,
    n_folds=5,
)
print("XGBoost CV Macro F1:", xgb_result["cv_mean_macro_f1"])


In [ ]:
# XGBoost artifact verification (Kenneth, Role 4)

assert (PATHS.models / "xgb_optuna.joblib").is_file(), "XGBoost model missing"
assert (PATHS.models / "xgb_optuna.meta.json").is_file(), "XGBoost meta missing"
print("XGBoost artifacts verified.")


In [ ]:
# TabNet training (Dwayne, Role 5)
# Contract: docs/contracts/role-5-training-execution-spec.md
# Estimated wall time: 6 to 8 hours

tabnet_result = train_tabnet(
    X_train=X_train,
    y_train=y_train,
    class_order=class_order,
    n_trials=20,
    n_folds=3,
)
print("TabNet CV Macro F1:", tabnet_result["cv_mean_macro_f1"])


In [ ]:
# TabNet artifact verification (Dwayne, Role 5)

assert (PATHS.models / "tabnet_best.zip").is_file(), "TabNet model missing"
assert (PATHS.models / "tabnet_best.meta.json").is_file(), "TabNet meta missing"
print("TabNet artifacts verified.")


In [ ]:
# Cross-model comparison (Kit, Role 1)
# Run after all three training cells complete.

rows = []

for name, meta_file in [
    ("Random Forest", "rf_baseline.meta.json"),
    ("XGBoost",       "xgb_optuna.meta.json"),
    ("TabNet",        "tabnet_best.meta.json"),
]:
    meta_path = PATHS.models / meta_file
    if not meta_path.is_file():
        rows.append({"model": name, "status": "MISSING"})
        continue
    m = json.load(open(meta_path))
    pc = m.get("per_class_recall_train", {})
    rows.append({
        "model":            name,
        "cv_mean_macro_f1": m["cv_mean_macro_f1"],
        "cv_std_macro_f1":  m["cv_std_macro_f1"],
        "u2r_recall":       pc.get("u2r"),
        "r2l_recall":       pc.get("r2l"),
        "probe_recall":     pc.get("probe"),
        "dos_recall":       pc.get("dos"),
        "normal_recall":    pc.get("normal"),
        "status":           "OK",
    })

df = pd.DataFrame(rows)
display(df)

comparison_path = PATHS.outputs / "02_training" / "model_comparison.csv"
comparison_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)


In [ ]:
# Final summary (Kit, Role 1)

print("=" * 68)
print("Notebook 02 Summary")
print("=" * 68)
for name, meta_file in [
    ("Random Forest", "rf_baseline.meta.json"),
    ("XGBoost",       "xgb_optuna.meta.json"),
    ("TabNet",        "tabnet_best.meta.json"),
]:
    meta_path = PATHS.models / meta_file
    if meta_path.is_file():
        m = json.load(open(meta_path))
        print(f"{name:<14} CV Macro F1 = {m['cv_mean_macro_f1']:.4f}")
    else:
        print(f"{name:<14} MISSING")
print("=" * 68)
